# 04 Grasp Scoring

Bin picking 中常常不需要完整识别物体，也可以根据点云选择合理抓取。

![Grasp scoring visual map](assets/figures/04_grasp_scoring.png)

## Learning Objectives

- Explain the difference between recognizing an object and selecting a feasible grasp.
- Read a simple antipodal grasp score from contact balance and gripper width.
- Connect grasp scoring errors to point cloud coverage and candidate generation.

## Checkpoint

- Run the scoring cell and identify why the balanced candidate wins.
- Explain what happens when a candidate is too narrow.
- Name one physical constraint missing from this simplified scorer.

## Practice Task

Create one new candidate grasp, predict its score qualitatively, then run the cell and compare your prediction with the output.

## Concept Map

![Colab concept image](assets/colab/04_grasp_scoring_concept.png)

**Concept image.** A grasp scorer turns local point support and gripper geometry into ranked grasp candidates.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


In [ ]:
import numpy as np
from rml.grasp_scoring import score_antipodal_grasps

points = np.array([[-0.5, 0.02], [-0.48, -0.02], [0.5, 0.01], [0.52, -0.01], [0.0, 0.6]])
candidates = [
    {"name": "off_center", "center": [0.0, 0.4], "angle": 0.0, "width": 1.0},
    {"name": "balanced", "center": [0.0, 0.0], "angle": 0.0, "width": 1.0},
    {"name": "too_narrow", "center": [0.0, 0.0], "angle": 0.0, "width": 0.4},
]
scores = score_antipodal_grasps(points, candidates)
[(s.name, round(s.score, 2), s.left_contacts, s.right_contacts) for s in scores]

## Result Figure

Show the point cloud next to a score chart so the ranking can be checked against contact evidence.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
fig, (ax_points, ax_scores) = plt.subplots(1, 2, figsize=(8.5, 3.6))
ax_points.scatter(points[:, 0], points[:, 1], s=85, label='points')
for candidate in candidates:
    center = np.asarray(candidate['center'], dtype=float)
    ax_points.scatter([center[0]], [center[1]], marker='x', s=90, label=candidate['name'])
ax_points.set_aspect('equal', adjustable='box')
ax_points.set_xlabel('closing axis')
ax_points.set_ylabel('approach axis')
ax_points.legend(frameon=False, fontsize=8)
names = [score.name for score in scores]
values = [score.score for score in scores]
ax_scores.bar(names, values, color=['#4C78A8', '#F58518', '#54A24B'])
ax_scores.set_ylabel('score')
ax_scores.tick_params(axis='x', rotation=25)
fig.tight_layout()
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Change the candidate gripper widths and print the resulting score ranking.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
for width in [0.4, 0.8, 1.0, 1.4]:
    width_candidates = [dict(candidate, width=width) for candidate in candidates]
    ranking = score_antipodal_grasps(points, width_candidates)
    print('width=', width, [(score.name, round(score.score, 2)) for score in ranking])

## Reflection Prompt

为什么 balanced candidate 比 off_center 更可靠？从左右接触数量和夹爪宽度两方面说明。

Exercise: 增加一个偏心候选，说明为什么它的接触更不均衡。